In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm
from PIL import Image
from transformers import AutoTokenizer, AutoProcessor, AutoModelForImageTextToText
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image_dog.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_dog.jpg
wget --no-verbose --output-document=image_me.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_me.jpg
wget --no-verbose --output-document=image_cc_1.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_cc_1.jpg

# Lecture 19 - Vision and Language

How text becomes tokens, how an image becomes (many more) tokens, and a small instruction-tuned vision-language model (SmolVLM, 256M parameters) that we can run ourselves, including its hallucinations.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

## Tokenisation

Language models read sub-word *tokens*, produced by byte-pair encoding (BPE): frequent character sequences become single tokens, rare words are split into pieces. We compare the GPT-2 tokenizer (50k tokens, 2019) with the GPT-4o tokenizer (200k tokens, 2024).

In [ ]:
tokenizers = {'GPT-2': AutoTokenizer.from_pretrained('openai-community/gpt2'),
              'GPT-4o': AutoTokenizer.from_pretrained('Xenova/gpt-4o')}
texts = ['Christian Rupprecht teaches computer vision in Oxford.',
         'Donaudampfschifffahrtsgesellschaftskapitän',
         'The answer is 3.14159 or 12345678.',
         'for i in range(10):\n    print(i)',
         'Η όραση υπολογιστών είναι διασκεδαστική.']

def show_tokens(ax, text, tokenizer, y):
    ids = tokenizer.encode(text)
    pieces = [tokenizer.decode([i]) for i in ids]
    x = 0
    for k, p in enumerate(pieces):
        label = p.replace('\n', '⏎').replace(' ', '·')
        t = ax.text(x, y, label, fontsize=11, family='monospace', va='center',
                    bbox=dict(boxstyle='square,pad=0.15', facecolor=plt.cm.Pastel1(k % 9), edgecolor='none'))
        x += 0.0095 * max(len(label), 1) + 0.006
    return len(ids)

fig, ax = plt.subplots(figsize=(18, 0.9 * len(texts) * len(tokenizers)))
y = 0
for text in texts:
    for name, tok in tokenizers.items():
        n = show_tokens(ax, text, tok, y)
        ax.text(-0.01, y, f'{name} ({n})', ha='right', va='center', fontsize=10)
        y -= 1
    y -= 0.5
ax.set_xlim(-0.15, 1.0)
ax.set_ylim(y, 1)
ax.axis('off')
ax.set_title('Tokens (number of tokens in brackets; · = space, ⏎ = newline)')
savefig('vl_tokenisation')
plt.show()
# GPT-2's vocabulary was built mostly from English text: Greek is split into single characters or even bytes
# (a byte that is only part of a character cannot be shown on its own and appears as �)

## A small vision-language model

Modern VLMs follow one recipe (Lecture 19): a vision encoder (here a SigLIP ViT) turns the image into patch features, a small projector maps them to the language model's token embeddings, and the language model reads them like text. SmolVLM splits large images into tiles of 512x512 pixels plus a downscaled global view, and turns each into 64 visual tokens.

In [ ]:
model_name = 'HuggingFaceTB/SmolVLM-256M-Instruct'
processor = AutoProcessor.from_pretrained(model_name)
vlm = AutoModelForImageTextToText.from_pretrained(model_name).to(device, torch.bfloat16 if device == 'cuda' else torch.float32).eval()
image_token = processor.tokenizer.convert_tokens_to_ids('<image>')

def ask(image, question, max_new_tokens=80, longest_edge=2048):
    # the processor resizes the image so that its longest side is `longest_edge` (a multiple of 512), then cuts it into 512x512 tiles
    messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': question}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image], return_tensors='pt', size={'longest_edge': longest_edge}).to(device)
    with torch.no_grad():
        out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer = processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0].strip()
    n_image = (inputs['input_ids'] == image_token).sum().item()
    return answer, n_image, inputs['input_ids'].shape[1] - n_image

dog = Image.open('image_dog.jpg').convert('RGB')
answer, n_image, n_text = ask(dog, 'Describe this image in one sentence.')
print(answer)
print(f'input: {n_image} image tokens and {n_text} text tokens')

In [ ]:
# an image is worth many words: the number of visual tokens grows with the resolution the model looks at
sizes = [512, 1024, 1536, 2048]
counts = [ask(dog, 'Hi', max_new_tokens=1, longest_edge=s)[1] for s in sizes]
plt.figure(figsize=(7, 4))
plt.plot(sizes, counts, 'o-')
for s, c in zip(sizes, counts):
    plt.text(s, c + 30, str(c), ha='center')
plt.ylim(0, max(counts) * 1.15)
plt.xlabel('longest side of the image as seen by the model (px)')
plt.ylabel('# visual tokens')
plt.title(f'SmolVLM: visual tokens vs. image size (the question "Hi" is {len(processor.tokenizer.encode("Hi"))} token)')
plt.grid(alpha=0.3)
savefig('vl_visual_tokens')
plt.show()

## Asking questions, and hallucination

The model answers by continuing the conversation. It is fluent, and often right, but it can also produce answers that are not supported by the image (*hallucination*). Leading questions (that presuppose something that is not there) and counting are classic failure cases. Try your own questions and images; larger models (e.g. `SmolVLM-500M-Instruct`, `Qwen2.5-VL-3B-Instruct`) are much better but need a GPU.

In [ ]:
questions = [(dog, 'What animal is this?'),
             (dog, 'What is the weather like?'),
             (dog, 'What colour is the frisbee the dog is holding?'),
             (dog, 'How many dogs are in the image?'),
             (Image.open('image_me.jpg').convert('RGB'), 'Where was this photo taken?'),
             (Image.open('image_me.jpg').convert('RGB'), 'How many people are in the image?'),
             (Image.open('image_cc_1.jpg').convert('RGB'), 'What is this building?')]
rows = []
for image, q in questions:
    rows.append((image, q, ask(image, q, max_new_tokens=40)[0]))

fig, axs = plt.subplots(len(rows), 1, figsize=(10, 1.5 * len(rows)))
for ax, (image, q, a) in zip(axs, rows):
    ax.imshow(np.asarray(image.resize((180, int(180 * image.height / image.width)))), extent=(0, 1.6, 0, 1.6 * image.height / image.width))
    ax.text(1.8, 0.75, f'Q: {q}', fontsize=14, va='center', weight='bold')
    ax.text(1.8, 0.3, f'A: {a}', fontsize=14, va='center')
    ax.set_xlim(0, 9)
    ax.set_ylim(0, 1.1)
    ax.axis('off')
plt.tight_layout()
savefig('vl_questions')
plt.show()
for _, q, a in rows:
    print(f'Q: {q}\nA: {a}\n')